In [2]:
# notebook for assignment for homework 2

In [3]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)
documents = [file.parse() for file in reader.read()]

In [4]:
data_gen_instructions = """
You emulate a student who is taking our LLM course.
You are given one lesson page from the course.
Formulate 5 questions this student might ask that are answered by this page.

Rules:
- The page should contain the answer to each question.
- Make the questions complete and not too short.
- Use as few words as possible from the page; don't copy its phrasing.
- The questions should resemble how people actually ask things online:
  not too formal, not too short, not too long.
- Ask about the content of the lesson, not about its formatting or filename.
""".strip()

In [5]:
pages = documents[:3]

In [6]:
import json

for page in pages:
    user_prompt = json.dumps({
        "filename": page["filename"],
        "content": page["content"]
    })

In [7]:
from openai import OpenAI
import os


openai_client = OpenAI(
    api_key=os.getenv("GROQ_API_KEY"),
    base_url="https://api.groq.com/openai/v1"
)

# list models available to your Groq API key
models = openai_client.models.list()
sorted([m.id for m in models.data])[:30]

['allam-2-7b',
 'canopylabs/orpheus-arabic-saudi',
 'canopylabs/orpheus-v1-english',
 'groq/compound',
 'groq/compound-mini',
 'llama-3.1-8b-instant',
 'llama-3.3-70b-versatile',
 'meta-llama/llama-4-scout-17b-16e-instruct',
 'meta-llama/llama-prompt-guard-2-22m',
 'meta-llama/llama-prompt-guard-2-86m',
 'openai/gpt-oss-120b',
 'openai/gpt-oss-20b',
 'openai/gpt-oss-safeguard-20b',
 'qwen/qwen3-32b',
 'qwen/qwen3.6-27b',
 'whisper-large-v3',
 'whisper-large-v3-turbo']

In [8]:
MODEL="openai/gpt-oss-120b"

In [9]:
from evaluation_utils import Questions, llm_structured
# from openai import OpenAI
# import os


# openai_client = OpenAI(
#     api_key=os.getenv("GROQ_API_KEY"),
#     base_url="https://api.groq.com/openai/v1"
# )

# # list models available to your Groq API key
# models = openai_client.models.list()
# sorted([m.id for m in models.data])[:30]

all_questions = []
token_usage = []

for page in pages:
    user_prompt = json.dumps({
        "filename": page["filename"],
        "content": page["content"]
    })

    questions, usage = llm_structured(
        client=openai_client,
        model="openai/gpt-oss-20b",
        instructions=data_gen_instructions,
        user_prompt=user_prompt,
        output_type=Questions
    )

    token_usage.append(usage.input_tokens)

    for q in questions.questions:
        all_questions.append({
            "question": q,
            "filename": page["filename"]
        })

In [10]:
avg_tokens = sum(token_usage) / len(token_usage)

print(avg_tokens)

1483.0


In [11]:
# ANS 1 -- 1483

In [12]:
import pandas as pd

ground_truth = pd.read_csv("ground-truth.csv").to_dict(orient="records")

In [13]:
from gitsource import chunk_documents

chunks = chunk_documents(
    documents,
    size=2000,
    step=1000
)

In [14]:
# TEXT SERACH FUNCTION

In [15]:
from minsearch import Index

text_index = Index(
    text_fields=["content"],
    keyword_fields=["filename"]
)

text_index.fit(chunks)

In [16]:
#  Create Search Function

def text_search(query, num_results=5):
    return text_index.search(
        query=query,
        filter_dict={},
        num_results=num_results
    )

In [17]:
q = ground_truth[0]["question"]
print(q)

What exactly is a retrieval-augmented generation system, and why does it help with answers that the model wouldn't know on its own?


In [18]:
results = text_search(q)

In [19]:
# Step 7 — Inspect Results
for r in results:
    print(r["filename"])



01-agentic-rag/lessons/03-rag.md
01-agentic-rag/lessons/13-function-calling.md
01-agentic-rag/lessons/03-rag.md
01-agentic-rag/lessons/13-function-calling.md
01-agentic-rag/lessons/01-intro.md


In [20]:
# VECTOR SEARCH

In [21]:
from minsearch import VectorSearch
from embedder import Embedder

# 1. Initialize your local embedder
embedder = Embedder()


texts_to_embed = [chunk['content'] for chunk in chunks]


vectors = embedder.encode_batch(texts_to_embed)


vector_index = VectorSearch()
vector_index.fit(vectors, chunks)

print("Vector search index successfully fitted!")


Vector search index successfully fitted!


In [22]:
def vector_search(query, num_results=5):
    query_vector=embedder.encode(query)
    return vector_index.search(
        query_vector,
        num_results=num_results
    )

In [23]:
q = ground_truth[0]["question"]

print(q)

What exactly is a retrieval-augmented generation system, and why does it help with answers that the model wouldn't know on its own?


In [24]:
vector_results = vector_search(q)

In [25]:
for r in results:
    print(r["filename"])

01-agentic-rag/lessons/03-rag.md
01-agentic-rag/lessons/13-function-calling.md
01-agentic-rag/lessons/03-rag.md
01-agentic-rag/lessons/13-function-calling.md
01-agentic-rag/lessons/01-intro.md


In [26]:
# HYBRID SEARCH

In [40]:
def rrf(result_lists, k=60, num_results=5):
    scores = {}
    docs = {}

    for results in result_lists:
        for rank, doc in enumerate(results):
            key = (doc["filename"], doc["start"])
            scores[key] = scores.get(key, 0) + 1 / (k + rank)
            docs[key] = doc

    ranked = sorted(scores, key=scores.get, reverse=True)
    return [docs[key] for key in ranked[:num_results]]

In [45]:
def hybrid_search(query, k=200):
    text_results = text_search(query, num_results=10)
    vector_results = vector_search(query, num_results=10)
    return rrf([text_results, vector_results], k=k)

In [29]:
import numpy as np

In [30]:
def compute_relevance(record, search_function):
    """
    record:
        {
            "question": "...",
            "filename": "..."
        }

    Returns something like:

    [1,0,0,0,0]

    or

    [0,0,1,0,0]
    """

    question = record["question"]
    expected_filename = record["filename"]

    results = search_function(question)

    relevance = []

    for doc in results:
        relevance.append(
            int(doc["filename"] == expected_filename)
        )

    return relevance

In [31]:
def hit_rate(relevance_total):
    """
    Percentage of questions where
    the correct document appears
    anywhere in the returned results.
    """

    cnt = 0

    for relevance in relevance_total:
        if True in relevance:
            cnt += 1

    return cnt / len(relevance_total)

In [32]:
# Mean Reciprocal Rank (MRR)

def mrr(relevance_total):
    """
    Mean Reciprocal Rank
    """

    total_score = 0.0

    for relevance in relevance_total:

        for rank, value in enumerate(relevance):

            if value == 1:
                total_score += 1 / (rank + 1)
                break

    return total_score / len(relevance_total)

In [33]:
def evaluate(ground_truth, search_function):
    """
    Evaluate a search function on the entire ground truth.
    """

    relevance_total = []

    for record in ground_truth:
        relevance = compute_relevance(record, search_function)
        relevance_total.append(relevance)

    return {
        "hit_rate": hit_rate(relevance_total),
        "mrr": mrr(relevance_total)
    }

In [34]:
text_metrics = evaluate(
    ground_truth,
    text_search
)

text_metrics

{'hit_rate': 0.7583333333333333, 'mrr': 0.5942592592592594}

In [35]:
vector_metrics = evaluate(
    ground_truth,
    vector_search
)

vector_metrics

{'hit_rate': 0.725, 'mrr': 0.5486111111111112}

In [46]:
hybrid_metrics = evaluate(
    ground_truth,
    hybrid_search
)

hybrid_metrics

{'hit_rate': 0.8361111111111111, 'mrr': 0.637916666666667}

In [47]:
import pandas as pd

results = pd.DataFrame([
    {
        "Method": "Text Search",
        **text_metrics
    },
    {
        "Method": "Vector Search",
        **vector_metrics
    },
    {
        "Method": "Hybrid Search",
        **hybrid_metrics
    }
])

results

,Method,hit_rate,mrr
0,Text Search,0.758333,0.594259
1,Vector Search,0.725000,0.548611
2,Hybrid Search,0.836111,0.637917
